In [1]:
import sys
import pandas as pd
from pathlib import Path

from pydantic import BaseModel

import os
from openai import OpenAI
from openai.lib._pydantic import to_strict_json_schema
from dotenv import load_dotenv  
from pprint import pprint

from tqdm import tqdm
import constants

from prompting_utils import create_system_prompt
from model_utils import from_series_to_basemodel

import json

from textwrap import dedent

from pathlib import Path


from time import perf_counter

# Preliminari

In [ ]:
# Configurazione OpenAI
load_dotenv()
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

# Parametri
base_dir = Path.cwd().parent
SYSTEM_PROMPT_FILE_NAME = constants.SYSTEM_PROMPT_3
TEMPERATURE = 0.0
BASELINE_INFERENCE = True
BASELINE_MODEL = constants.OPENAI_GPT_4_1_NANO
BASELINE_RESULTS_FILE_NAME = 'results_2_' + BASELINE_MODEL + '.jsonl'

PYDANTIC_MODEL = constants.RectalCancerStagingData

#Carica system prompt da file
system_prompt_path = base_dir / "data" / "prompts" / SYSTEM_PROMPT_FILE_NAME
system_prompt = create_system_prompt(system_prompt_path, PYDANTIC_MODEL)

In [3]:
if True:
    print(system_prompt)

Sei un assistente medico radiologico esperto nella stadiazione del carcinoma rettale tramite Risonanza Magnetica (RM).

Il tuo compito è estrarre le caratteristiche strutturate dal referto RM fornito e mappare i dati nello schema JSON fornito.

Regole di output rigorose:

1. Restituisci esclusivamente un oggetto JSON valido. Nessun testo introduttivo, spiegazione, codice o commento fuori dal JSON.
2. Rispetta esattamente i tipi e i valori consentiti per ciascun campo.
4. Se un campo richiede valori enumerati, usa esclusivamente uno dei valori ammessi, senza mai usare sinonimi o varianti.
5. Se un campo è numerico ma nel testo non compare un valore, scrivi null, come indicato nello schema.
6. Non aggiungere o inventare informazioni non presenti nel referto.

Ecco dei criteri e consigli di estrazione per alcuni dei campi che dovrai estrarre:

Morfologia:
  - Se il referto menziona componenti aggettanti, vegetanti o endoluminali, classifica come solido_polipoide.
  - Se si parla di ispess

# Load Data

In [4]:
# Carichiamo i nostri file csv
file_names = {
    'validation': constants.VALIDATION_SPLIT_FILE_NAME,
    'test': constants.TEST_SPLIT_FILE_NAME,
}

paths = {split: Path('../data/' + file_name) for split, file_name in file_names.items()}

data = dict()
for split, path in paths.items():
    data[split] = pd.read_csv(path)

validation_data, test_data = data['validation'], data['test']

################################
# Convert float columns to Int64
################################
float_cols = test_data.select_dtypes("float").columns
for col in float_cols:
    test_data[col] = test_data[col].round().astype("Int64")
    validation_data[col] = validation_data[col].round().astype("Int64")
    
# Check duplicatest
assert set(test_data.id) & set(validation_data.id) == set(), "There are overlapping IDs between test and validation sets!"

print(f"{len(test_data) = }")
print(f"{len(validation_data) = }")

len(test_data) = 67
len(validation_data) = 63


# Generazione

## Preliminari generazione

In [5]:
# Funzione generatrice
def extract_data_from_report(model: str,
                             report_text: str,
                             system_prompt: str = system_prompt,
                             temperature: float = TEMPERATURE,
                             output_format: type[BaseModel] = PYDANTIC_MODEL):
    response = client.responses.parse(
        model=model,
        temperature=temperature,
        input=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': report_text},
            
        ],
        text_format=output_format
    )
    return response

In [6]:
# Esempio
if False:
    result = extract_data_from_report(BASELINE_MODEL, data['test'].iloc[0]['report_text'])

In [7]:
if False: # To see the full output
    pprint(result.model_dump())
if False:  # To see the string output text
    print(result.output_text)
if False:  # To see the parsed output as a pydantic BaseModel
    print(result.output_parsed)
if False:
    print(result.output_parsed.model_dump(mode='json'))

## Baseline inference
Usiamo modelli non finetunati. Solo prompt engineering.

In [19]:
import time
time.sleep(3)

In [ ]:
print(BASELINE_MODEL)
if BASELINE_INFERENCE:
    df = pd.concat([validation_data, test_data], ignore_index=True)
    ids = []
    actual = []
    predicted = []
    splits = []
    for i in tqdm(range(df.shape[0])):
        row = df.iloc[i]
        splits.append(row['split'])
        output = extract_data_from_report(model=BASELINE_MODEL, report_text=row[constants.REPORT_COLUMN_NAME])
        time.sleep(10)
        real = from_series_to_basemodel(row, PYDANTIC_MODEL)
        ids.append(row[constants.ID_COMULM_NAME])
        actual.append(real.model_dump(mode='json'))
        if output is None:
            predicted.append('no output')
        else:
            predicted.append(output.output_parsed.model_dump(mode='json'))

gpt-4.1-2025-04-14


100%|██████████| 10/10 [02:37<00:00, 15.71s/it]


In [41]:
if BASELINE_INFERENCE:
    results_dicts = []
    assert len(ids) == len(actual) == len(predicted)
    for id, act, pred, split in zip(ids, actual, predicted, splits):
        results_dicts.append(
            {
                'model': BASELINE_MODEL,
                'split': split,
                'id': int(id),
                'actual': act,
                'prediction': pred
            }
        )
    # Salvataggio
    with open(base_dir / 'data' / 'inference' / BASELINE_RESULTS_FILE_NAME, 'w', encoding='utf-8') as f:
        for r in results_dicts:
            f.write(json.dumps(r) + '\n')

In [42]:
output.model_dump()

{'id': 'resp_0164f4d4d4ef35e000698be3f07f4481a2a0121edabf28e8aa',
 'created_at': 1770775536.0,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'gpt-4.1-2025-04-14',
 'object': 'response',
 'output': [{'id': 'msg_0164f4d4d4ef35e000698be3f0edc081a290f733e91551a990',
   'content': [{'annotations': [],
     'text': '{\n  "morfologia": "solido_anulare",\n  "ore_inizio": null,\n  "ore_fine": null,\n  "spessore_parietale": null,\n  "estensione_cranio_caudale": 40,\n  "distanza_oai": 60,\n  "posizione": {\n    "basso": "no",\n    "medio": "si",\n    "alto": "si",\n    "giunzione": "no"\n  },\n  "riflessione_peritoneale_anteriore": "cavallo",\n  "infiltrazione_tessuto_adiposo": "si_5mm_plus",\n  "infiltrazione_sfinteri": "no",\n  "infiltrazione_organi_extra": "no",\n  "infiltrazione_organi_dettagli": {\n    "pavimento_pelvico": "no",\n    "altro": "no"\n  },\n  "coinvolgimento_riflessione_peritoneale": "si",\n  "coinvolgimento_fascia_mesorettale": 